# PySpark 15: interview concepts

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-15-interview-concepts.ipynb)

Interviewers rarely ask you to recite an API. They ask why a job behaves the way it does, what you would change, and
how you would prove it worked. This lesson answers the questions that come up most, grouped by topic, in the form a
good interview answer takes. Say each answer aloud before reading it, then follow the link to the lesson that shows it
running.

**Prerequisites:** lessons [1 to 14](pyspark-00-start-here.ipynb). For hands-on practice, finish with the
[PySpark coding challenges](pyspark-coding-challenges.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

## How to structure an answer

A strong answer names three things: the **API** you would use, what the **engine** does with it, and the
**production concern** it raises. For "How do you deduplicate orders?":

| part | what to say |
| --- | --- |
| API | `row_number()` over `Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))`, keep `rn = 1` |
| engine | one shuffle by `order_id`, a sort within each partition, then one pass |
| production | deterministic tie-breaker so reruns pick the same row; merge into the target for idempotency; watch for a hot key skewing the window |

Tie every method to a shuffle, a partition count or a failure mode. Naming a configuration setting is much weaker than
naming the evidence that justifies it.

## Architecture and execution

How Spark runs code.

**Q. What runs on the driver and what runs on the executors?**

The driver runs your program, builds and optimizes the plan, schedules tasks, and receives anything you collect. Executors run tasks, one per core at a time, and hold partitions, shuffle files, cached blocks and broadcast tables. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. What is lazy evaluation, and why does Spark use it?**

Transformations only add steps to a plan; an action runs it. Waiting lets Catalyst optimize the whole query: combine filters, push them to the reader, drop unused columns and pick join strategies. The cost is that errors surface at the action, and each action recomputes unless you cache. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. What is the difference between narrow and wide transformations?**

Narrow ones (`select`, `filter`, `withColumn`) work within a partition and stay in the same stage. Wide ones (`groupBy`, `join`, `distinct`, `orderBy`, windows with `partitionBy`) need rows with the same key together, so they shuffle and start a new stage. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. How do jobs, stages and tasks relate?**

Each action starts one or more jobs; a job is cut into stages at every shuffle; a stage runs one task per partition. A stage finishes when its slowest task does. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. What do Catalyst and Tungsten do?**

Catalyst analyzes and optimizes the logical plan with rules (pushdown, pruning, constant folding) and picks a physical plan; Tungsten runs it efficiently with a compact binary row format and generated code. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. Why prefer DataFrames to RDDs?**

Spark can see and optimize DataFrame expressions; RDD lambdas are opaque and move every row between the JVM and Python. Spark Connect does not support RDDs at all. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

**Q. What is Spark Connect?**

A client-server mode where a thin client sends plans over gRPC to a remote driver. DataFrame code is unchanged; RDDs, `sparkContext` and JVM internals are not available. *More in [lesson 1](pyspark-01-how-spark-runs.ipynb).*

## DataFrames, types and data

Everyday API questions with a trap in each.

**Q. Why prefer `unionByName` to `union`?**

`union` matches columns by position, so frames listing columns in different orders are silently mixed. `unionByName` matches by name and can fill missing columns. *More in [lesson 2](pyspark-02-dataframe-essentials.ipynb).*

**Q. Why does `df.where(col != 'x')` drop rows with nulls?**

A comparison with null is unknown, and a filter keeps only true rows. Add `| col.isNull()` when nulls should stay. *More in [lesson 2](pyspark-02-dataframe-essentials.ipynb).*

**Q. Why store money as `decimal` instead of `double`?**

Doubles are binary approximations: 0.1 + 0.2 is 0.30000000000000004, and errors add up over millions of rows. Decimals are exact. *More in [lesson 3](pyspark-03-types-functions-and-nulls.ipynb).*

**Q. What changed about casting in Spark 4?**

ANSI SQL mode is on by default: invalid casts, overflow and division by zero raise errors instead of returning null. Use `try_cast` and other `try_` functions where bad values are expected, then count the nulls. *More in [lesson 3](pyspark-03-types-functions-and-nulls.ipynb).*

**Q. Why give a schema when reading CSV or JSON?**

Inference reads the data an extra time and can guess different types per delivery. A schema is faster and enforces the contract. *More in [lesson 4](pyspark-04-reading-and-writing-data.ipynb).*

**Q. How do you handle malformed records?**

`PERMISSIVE` mode with `columnNameOfCorruptRecord` keeps bad rows for a quarantine; `FAILFAST` stops the job; `DROPMALFORMED` hides data loss. Reconcile landed rows with parsed plus rejected rows, and do not trust a bare `count()`, which skips parsing. *More in [lesson 4](pyspark-04-reading-and-writing-data.ipynb).*

**Q. What is partition pruning?**

Skipping partition folders whose values cannot match a filter, visible as `PartitionFilters` in the plan. Dynamic partition pruning does the same at runtime from a join's filtered side. *More in [lesson 4](pyspark-04-reading-and-writing-data.ipynb).*

**Q. What is the difference between a managed and an external table?**

Dropping a managed table deletes its data; dropping an external table removes only the definition. *More in [lesson 5](pyspark-05-spark-sql-and-the-catalog.ipynb).*

**Q. How do you pass values into Spark SQL safely?**

With parameter markers: `spark.sql("... WHERE status = :status", args={"status": value})`. Formatting values into the SQL string invites injection. *More in [lesson 5](pyspark-05-spark-sql-and-the-catalog.ipynb).*

## Joins, aggregations and windows

Correctness first, then cost.

**Q. Which join returns customers who never ordered?**

`left_anti`. `left_semi` returns those who did order, without duplicating them. *More in [lesson 6](pyspark-06-joins.ipynb).*

**Q. Revenue doubled after a join. What happened?**

The dimension key was not unique, so each fact matched several rows. Check uniqueness before joining and row counts after. *More in [lesson 6](pyspark-06-joins.ipynb).*

**Q. Why do rows with null keys disappear from a join?**

`NULL = NULL` is not true. Use `eqNullSafe` when nulls should match. *More in [lesson 6](pyspark-06-joins.ipynb).*

**Q. How does a broadcast join work, and when does it fail?**

The small side is copied to every executor and the large side is joined without a shuffle. Spark broadcasts below `spark.sql.autoBroadcastJoinThreshold` (10 MB) or on a hint. If the side is not really small, the driver and executors run out of memory. *More in [lesson 6](pyspark-06-joins.ipynb).*

**Q. Sort-merge join versus shuffled hash join?**

Both shuffle both sides by the key. Sort-merge then sorts and merges, scaling to any size and spilling gracefully; shuffled hash builds a hash table per partition, avoiding the sort when one side is small enough. *More in [lesson 6](pyspark-06-joins.ipynb).*

**Q. Why do `sum` and `count` shuffle less than `countDistinct`?**

They are combined in a partial aggregate before the shuffle, so only one row per key and task moves. Distinct values cannot be combined that way. *More in [lesson 7](pyspark-07-aggregations-and-windows.ipynb).*

**Q. How do you tell a rollup subtotal from a real null key?**

With `grouping_id()` or `grouping(col)`. *More in [lesson 7](pyspark-07-aggregations-and-windows.ipynb).*

**Q. `row_number`, `rank` and `dense_rank` on 300, 300, 200?**

1, 2, 3; 1, 1, 3; and 1, 1, 2. *More in [lesson 7](pyspark-07-aggregations-and-windows.ipynb).*

**Q. Why does `last(col)` over `orderBy(ts)` return the current row?**

The default frame with an order runs from the partition's start to the current row. Use `rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)`. *More in [lesson 7](pyspark-07-aggregations-and-windows.ipynb).*

**Q. How do you find the longest streak of consecutive days?**

Gaps and islands: number each key's distinct days with `row_number`, subtract from the date, and group by the result; each group is a run. *More in [lesson 7](pyspark-07-aggregations-and-windows.ipynb).*

## Nested data and Python

Where silent data loss and slow jobs hide.

**Q. Which rows does `explode` drop?**

Rows whose array is null or empty. `explode_outer` keeps them with a null element. *More in [lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb).*

**Q. Why use `transform` or `aggregate` instead of explode and `groupBy`?**

Higher-order functions work inside each row: no shuffle, and one row per input row. *More in [lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb).*

**Q. Why are Python UDFs slow?**

Rows are serialized to Python workers and back, and the optimizer cannot see inside the function. Prefer built-ins, then pandas UDFs or Arrow UDFs. *More in [lesson 9](pyspark-09-python-udfs-and-pandas.ipynb).*

**Q. A UDF returns only nulls. Why?**

Its declared return type does not match what it returns; mismatches become nulls without an error. *More in [lesson 9](pyspark-09-python-udfs-and-pandas.ipynb).*

**Q. When do you use `applyInPandas` versus `mapInPandas`?**

`applyInPandas` when logic needs all rows of a group together (it shuffles by key); `mapInPandas` for batch-by-batch processing with no grouping. *More in [lesson 9](pyspark-09-python-udfs-and-pandas.ipynb).*

## Pipelines and streaming

Guarantees under retries and late data.

**Q. How do you make a batch job idempotent?**

Scope writes to what the batch owns: dynamic partition overwrite or a merge by key. Make deduplication deterministic and keep the clock out of the data. Prove it by running the batch twice. *More in [lesson 10](pyspark-10-batch-pipelines.ipynb).*

**Q. How do you keep the latest record per key from a change feed?**

`row_number` over the source sequence with a stable tie-breaker, keep `rn = 1`, then merge with a condition that only lets a newer version win. *More in [lesson 10](pyspark-10-batch-pipelines.ipynb).*

**Q. How do you build an SCD type 2 dimension?**

Compare incoming attributes with the current row null-safely; close changed rows, open new ones, insert new keys; keep exactly one current row per key. On Delta it is one `MERGE`. *More in [lesson 10](pyspark-10-batch-pipelines.ipynb).*

**Q. How does Structured Streaming achieve exactly-once results?**

A replayable source with offsets, a checkpoint of offsets and state, and an idempotent or transactional sink. *More in [lesson 11](pyspark-11-structured-streaming.ipynb).*

**Q. What is a watermark?**

How late data may arrive: the latest event time seen minus a delay. Windows that end before the watermark are final, and their state is dropped. *More in [lesson 11](pyspark-11-structured-streaming.ipynb).*

**Q. What does a stream-stream join need?**

Watermarks on both sides and a time-range condition, so buffered rows can be dropped. *More in [lesson 11](pyspark-11-structured-streaming.ipynb).*

## Performance and operations

Evidence before configuration.

**Q. How do you debug a slow job?**

Find the slowest stage, compare its maximum and median task times, read shuffle bytes, spill and GC time, and read the plan for unexpected exchanges, missing pruning and joins that should broadcast. Change one thing and measure again. *More in [lesson 12](pyspark-12-internals-and-monitoring.ipynb).*

**Q. One task takes far longer than the rest. What do you do?**

That is skew. Try AQE skew join, then broadcast the small side, salt the hot key, or fix its source (often a null or placeholder key). *More in [lesson 13](pyspark-13-performance-tuning.ipynb).*

**Q. How many shuffle partitions should a job use?**

`ceil(shuffle_bytes / ~128 MB)`; with AQE on, start generously and let it coalesce toward the advisory size. *More in [lesson 13](pyspark-13-performance-tuning.ipynb).*

**Q. `repartition` or `coalesce`?**

`repartition` shuffles to balance and can raise the count; `coalesce` only merges partitions, cannot raise the count, and keeps imbalance. *More in [lesson 13](pyspark-13-performance-tuning.ipynb).*

**Q. When should you cache?**

When several actions reuse an expensive DataFrame that fits in memory. Unpersist afterwards. Checkpoint instead when the lineage itself is the problem. *More in [lesson 12](pyspark-12-internals-and-monitoring.ipynb).*

**Q. How do you size executors?**

Three to five cores and 8 to 32 GB each is the usual range; memory overhead is added to the heap; total parallelism is executors times cores. *More in [lesson 12](pyspark-12-internals-and-monitoring.ipynb).*

**Q. Client or cluster deploy mode for a production job?**

Cluster mode, so the driver runs in the cluster and does not depend on the scheduler host. *More in [lesson 14](pyspark-14-production-pyspark.ipynb).*

## Key takeaways

- Answer with the API, the engine's behaviour and the production concern.
- Correctness traps (nulls, duplicates, `explode`, `union`, casts) come up as often as performance questions.
- For tuning questions, name the evidence before the fix, and how you measured the result.

**Next:** the [PySpark coding challenges](pyspark-coding-challenges.ipynb), then the
[end-to-end capstone](pyspark-concepts-end-to-end.ipynb).